# Linear — 45s motion design reel

Renders the reel on a Colab T4 and commits the finished MP4 back to the repo.

**Why this exists.** The pipeline is a pure-Python frame generator: each frame is a function
of its index, rasterised by `rsvg-convert`, then encoded by ffmpeg. No browser, no GPU, no
framework. That makes it embarrassingly parallel across CPU cores, and a T4 instance has far
more of them than a laptop — so the whole 1350-frame render lands in a couple of minutes.

**Requirements.** Colab ships Python and ffmpeg but none of the text or SVG stack, so the first
cell installs `librsvg2-bin` and Pango introspection. Pango matters for correctness, not just
convenience: it is the same shaper librsvg uses internally, so glyph positions computed here
agree with what actually gets drawn.

In [ ]:
!git clone https://github.com/emilio-n8n/linear-motion-reel.git
%cd linear-motion-reel
!bash tools/setup_env.sh

## Fetch the latest source

Re-run this cell to pick up commits pushed from elsewhere. `--depth 1` because the render
does not need history.

In [ ]:
%cd linear-motion-reel
!git pull --ff-only origin main

## Render

`--jobs` is capped below the core count on purpose: `rsvg-convert` is single-threaded per frame,
and oversubscribing every core makes the VM thrash rather than speed up. On a T4, 4 workers is
the sweet spot.

Frames are cached in `.build/frames/`, so if the cell is interrupted, re-running it resumes
from the first missing frame instead of starting over.

In [ ]:
import os, subprocess

CORES = os.cpu_count() or 4
JOBS = max(2, min(4, CORES - 1))
print(f"{CORES} cores, rendering on {JOBS} workers")

subprocess.run(
    ["python3", "src/render.py", "--width", "1920", "--jobs", str(JOBS)],
    check=True,
)

## Encode

CRF 17 at `-preset slow`. The grain layer is high-frequency noise, so a low bitrate shows it as
blocking; 1080p30 lands around 12–20 MB at this setting.

In [ ]:
subprocess.run(
    ["python3", "src/encode.py", "--crf", "17", "--preset", "slow"],
    check=True,
)

In [ ]:
from IPython.display import Video, display

display(Video("out/linear-craft-45s.mp4", embed=True, width=960))

## Commit the master back

This is the hand-off: the MP4 is committed to `main`, so it can be pulled locally with
`git pull` and lands in `out/`.

Configure the token once per runtime:

```python
from google.colab import userdata
token = userdata.get('GH_TOKEN')  # Colab secret, needs repo scope
```

In [ ]:
import os, subprocess
from datetime import datetime, timezone

TOKEN = os.environ.get("GH_TOKEN", "")
if not TOKEN:
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
    print("using GH_TOKEN from Colab secrets")

REPO = "https://github.com/emilio-n8n/linear-motion-reel.git"

# GitHub rejects pushes over ~100 MB; the master is far under that, but the
# intermediate frames are not, so only out/ is ever staged.
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

subprocess.run(["git", "config", "user.name", "colab-render"], check=True)
subprocess.run(["git", "config", "user.email", "colab-render@users.noreply.github.com"], check=True)
subprocess.run(["git", "remote", "set-url", "origin", f"https://x-access-token:{TOKEN}@{REPO}"], check=True)

subprocess.run(["git", "add", "out/linear-craft-45s.mp4"], check=True)
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout

if not status.strip():
    print("MP4 unchanged — nothing to commit")
else:
    size_mb = os.path.getsize("out/linear-craft-45s.mp4") / 1e6
    subprocess.run(
        ["git", "commit", "-m", f"render: 45s master ({size_mb:.1f} MB) {stamp}"],
        check=True,
    )
    subprocess.run(["git", "push", "origin", "main"], check=True)
    print(f"pushed {size_mb:.1f} MB")

## Pull it locally

On the machine with the repo:

```bash
git pull origin main
open out/linear-craft-45s.mp4
```

Or fetch just the file without touching the working tree:

```bash
git fetch origin main
git show origin/main:out/linear-craft-45s.mp4 > out/linear-craft-45s.mp4
```